In [19]:
X_test

array([[ 2.        , 19.        ,  2.        ,  0.07146735,  1.        ],
       [ 0.        , 10.        ,  3.        ,  0.3440715 ,  0.        ],
       [ 0.        ,  2.        ,  6.        ,  0.4450377 ,  1.        ],
       [ 0.        ,  8.        ,  2.        ,  0.65983887,  1.        ],
       [ 1.        , 14.        ,  6.        ,  0.2854929 ,  1.        ]])

In [18]:
model.tree.apply(X_test)

array([22,  4, 11, 12,  5])

In [20]:
model.get_leaf_stats(X_test)

[{'n': 563,
  'clicks': 86,
  'alpha': 87.0,
  'beta': 497.0,
  'used_global_fallback': False},
 {'n': 3739,
  'clicks': 439,
  'alpha': 440.0,
  'beta': 3320.0,
  'used_global_fallback': False},
 {'n': 863,
  'clicks': 148,
  'alpha': 149.0,
  'beta': 735.0,
  'used_global_fallback': False},
 {'n': 492,
  'clicks': 139,
  'alpha': 140.0,
  'beta': 373.0,
  'used_global_fallback': False},
 {'n': 867,
  'clicks': 52,
  'alpha': 53.0,
  'beta': 835.0,
  'used_global_fallback': False}]

In [23]:
model._leaf_ids(X_test)

array([22,  4, 11, 12,  5])

In [32]:

model = ActionTreeThompsonModel(
    max_depth=4,
    min_samples_leaf=300,
    alpha0=1.0,
    beta0=1.0,   # prior под низкий CTR
    c_min=5,
    random_state=1,
)
model.fit(X_test, y_test)

In [37]:
model.leaf_stats

{0: {'n': 5, 'clicks': 2, 'alpha': 3.0, 'beta': 4.0}}

In [36]:
model.sample_proba(X_test)

array([0.46124869, 0.54982507, 0.56647055, 0.40318186, 0.38091557])

In [31]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier

class ActionTreeThompsonModel:
    def __init__(
        self,
        max_depth=4,
        min_samples_leaf=300,
        alpha0=1.0,
        beta0=1.0,
        c_min=5,
        random_state=42,
    ):
        self.max_depth = max_depth
        self.min_samples_leaf = min_samples_leaf
        self.alpha0 = float(alpha0)
        self.beta0 = float(beta0)
        self.c_min = int(c_min)
        self.random_state = random_state

        self.tree = None
        self.leaf_stats = {}
        self.global_alpha = None
        self.global_beta = None

    def fit(self, X, y):
        X = np.asarray(X)
        y = np.asarray(y).astype(int)

        self.tree = DecisionTreeClassifier(
            criterion="log_loss",
            max_depth=self.max_depth,
            min_samples_leaf=self.min_samples_leaf,
            random_state=self.random_state,
        )
        self.tree.fit(X, y)

        leaf_ids = self.tree.apply(X)

        total_clicks = int(y.sum())
        total_n = int(len(y))
        self.global_alpha = self.alpha0 + total_clicks
        self.global_beta = self.beta0 + (total_n - total_clicks)

        self.leaf_stats = {}
        for leaf in np.unique(leaf_ids):
            mask = leaf_ids == leaf
            n = int(mask.sum())
            c = int(y[mask].sum())

            self.leaf_stats[int(leaf)] = {
                "n": n,
                "clicks": c,
                "alpha": self.alpha0 + c,
                "beta": self.beta0 + (n - c),
            }
        return self

    def _ensure_fitted(self):
        if self.tree is None:
            raise RuntimeError("Model is not fitted yet.")

    def _to_2d(self, X):
        X = np.asarray(X)
        if X.ndim == 1:
            X = X.reshape(1, -1)
        return X

    def _leaf_ids(self, X):
        self._ensure_fitted()
        X = self._to_2d(X)
        return self.tree.apply(X)

    def _effective_stats_for_leaf(self, leaf_id):
        leaf_id = int(leaf_id)
        stats = self.leaf_stats.get(leaf_id)

        if stats is None:
            print("KEK")
            return {
                "n": 0,
                "clicks": 0,
                "alpha": self.global_alpha,
                "beta": self.global_beta,
                "used_global_fallback": True,
            }

        if stats["clicks"] < self.c_min:
            return {
                "n": stats["n"],
                "clicks": stats["clicks"],
                "alpha": self.global_alpha,
                "beta": self.global_beta,
                "used_global_fallback": True,
            }

        return {
            "n": stats["n"],
            "clicks": stats["clicks"],
            "alpha": stats["alpha"],
            "beta": stats["beta"],
            "used_global_fallback": False,
        }

    def get_leaf_stats(self, X):
        leaf_ids = self._leaf_ids(X)
        return [self._effective_stats_for_leaf(leaf) for leaf in leaf_ids]

    def predict_mean(self, X):
        stats = self.get_leaf_stats(X)
        return np.array([
            s["alpha"] / (s["alpha"] + s["beta"])
            for s in stats
        ])

    def sample_proba(self, X, n_samples=1, random_state=None):
        rng = np.random.default_rng(random_state)
        stats = self.get_leaf_stats(X)

        alpha = np.array([s["alpha"] for s in stats], dtype=float)
        beta = np.array([s["beta"] for s in stats], dtype=float)

        if n_samples == 1:
            return rng.beta(alpha, beta)

        return rng.beta(
            np.broadcast_to(alpha, (n_samples, len(alpha))),
            np.broadcast_to(beta, (n_samples, len(beta))),
        )

    def update_one(self, x, reward):
        self._ensure_fitted()
        reward = int(reward)

        leaf = int(self._leaf_ids(x)[0])

        if leaf not in self.leaf_stats:
            self.leaf_stats[leaf] = {
                "n": 0,
                "clicks": 0,
                "alpha": self.alpha0,
                "beta": self.beta0,
            }

        self.leaf_stats[leaf]["n"] += 1
        self.leaf_stats[leaf]["clicks"] += reward
        self.leaf_stats[leaf]["alpha"] += reward
        self.leaf_stats[leaf]["beta"] += (1 - reward)

        self.global_alpha += reward
        self.global_beta += (1 - reward)

    def update_batch(self, X, y):
        self._ensure_fitted()
        X = self._to_2d(X)
        y = np.asarray(y).astype(int)

        leaf_ids = self.tree.apply(X)
        for leaf, reward in zip(leaf_ids, y):
            leaf = int(leaf)

            if leaf not in self.leaf_stats:
                self.leaf_stats[leaf] = {
                    "n": 0,
                    "clicks": 0,
                    "alpha": self.alpha0,
                    "beta": self.beta0,
                }

            self.leaf_stats[leaf]["n"] += 1
            self.leaf_stats[leaf]["clicks"] += int(reward)
            self.leaf_stats[leaf]["alpha"] += int(reward)
            self.leaf_stats[leaf]["beta"] += int(1 - reward)

        self.global_alpha += int(y.sum())
        self.global_beta += int(len(y) - y.sum())


def generate_synthetic_action_data(n=10000, random_state=42):
    """
    Данные для одного action:
    X -> контекст
    y -> был клик или нет именно на этом action
    p_true -> истинная вероятность клика
    """
    rng = np.random.default_rng(random_state)

    platform = rng.integers(0, 3, size=n)          # 0=iOS, 1=Android, 2=Web
    hour = rng.integers(0, 24, size=n)
    prev_impressions = rng.poisson(3, size=n)
    affinity = rng.beta(2, 5, size=n)
    is_premium = rng.integers(0, 2, size=n)

    logit = (
        -3.0
        + 0.7 * (platform == 0)
        + 0.35 * (platform == 1)
        + 0.8 * (hour >= 18)
        - 0.15 * prev_impressions
        + 2.5 * affinity
        + 0.45 * is_premium
        - 0.7 * ((platform == 2) & (hour < 8))
    )
    p_true = 1.0 / (1.0 + np.exp(-logit))
    y = rng.binomial(1, p_true)

    X = np.column_stack([
        platform,
        hour,
        prev_impressions,
        affinity,
        is_premium,
    ])

    feature_names = [
        "platform",
        "hour",
        "prev_impressions",
        "affinity",
        "is_premium",
    ]
    return X, y, p_true, feature_names




In [2]:
# 1) генерим train
X_train, y_train, p_train, feature_names = generate_synthetic_action_data(
    n=12000,
    random_state=1,
)


In [6]:
p_train

array([0.173181  , 0.08709038, 0.10359359, ..., 0.10007945, 0.08821257,
       0.11511762])

In [7]:
# 2) обучаем дерево + alpha/beta в листьях
model = ActionTreeThompsonModel(
    max_depth=4,
    min_samples_leaf=300,
    alpha0=1.0,
    beta0=20.0,   # prior под низкий CTR
    c_min=5,
    random_state=1,
)
model.fit(X_train, y_train)

print("Feature names:", feature_names)
print("Train CTR:", y_train.mean())
print("Global posterior mean:",
      model.global_alpha / (model.global_alpha + model.global_beta))
print("Number of leaves with stats:", len(model.leaf_stats))

Feature names: ['platform', 'hour', 'prev_impressions', 'affinity', 'is_premium']
Train CTR: 0.132
Global posterior mean: 0.13185259129856086
Number of leaves with stats: 13


In [8]:
# 3) тест
X_test, y_test, p_test, _ = generate_synthetic_action_data(
    n=5,
    random_state=2,
)

print("\nPosterior means:")
print(model.predict_mean(X_test))

print("\nThompson samples:")
print(model.sample_proba(X_test, random_state=123))

print("\nTrue probs:")
print(p_test)


Posterior means:
[0.1489726  0.11702128 0.16855204 0.27290448 0.05968468]

Thompson samples:
[0.12883861 0.12271505 0.16419075 0.28219881 0.07254557]

True probs:
[0.13338772 0.13126485 0.16281837 0.37744601 0.08422586]


In [13]:
# 4) посмотрим leaf stats для первого объекта
leaf_id = int(model._leaf_ids(X_test[:1])[0])
print("\nRaw leaf stats for first test object:")
print("leaf_id =", leaf_id)
print(model.leaf_stats[leaf_id])


Raw leaf stats for first test object:
leaf_id = 22
{'n': 563, 'clicks': 86, 'alpha': 87.0, 'beta': 497.0}


In [ ]:


print("\nEffective leaf stats for first test object:")
print(model.get_leaf_stats(X_test[:1])[0])

# 5) онлайн-дообучение по одному примеру
before = model.predict_mean(X_test[:1])[0]
reward = int(y_test[0])

model.update_one(X_test[0], reward)

after = model.predict_mean(X_test[:1])[0]
print("\nOnline update:")
print("reward =", reward)
print("mean before =", before)
print("mean after  =", after)

# 6) онлайн-дообучение батчом
model.update_batch(X_test[1:], y_test[1:])
print("\nGlobal posterior mean after batch update:",
      model.global_alpha / (model.global_alpha + model.global_beta))